In [1]:
import json
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd


def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def normalize_status(x):
    if x == "PASS":
        return "PASS"
    if isinstance(x, str) and x.startswith("EXEC_FAIL"):
        return "EXEC_FAIL"
    if isinstance(x, str) and x.startswith("TEST_FAIL"):
        return "TEST_FAIL"
    return x


def analyze_run(run_dir):
    run_dir = Path(run_dir)

    step_df = read_jsonl(run_dir / "step_logs.jsonl")
    traj_df = read_jsonl(run_dir / "trajectory_logs.jsonl")
    summary = read_json(run_dir / "summary.json")

    method = summary.get("method", run_dir.name)
    total_problems = summary["total_problems"]
    max_calls = summary["max_calls"]

    print("=" * 80)
    print(f"Run: {method}")
    print("=" * 80)

    # 1. Overall summary
    overall = {
        "method": method,
        "total_problems": total_problems,
        "num_success": summary["num_success"],
        "success_at_k": summary["success_at_k"],
        "execution_success_rate": summary["execution_success_rate"],
        "conditional_success": summary["conditional_success"],
        "avg_calls": summary.get("avg_calls"),
        "avg_tokens": summary.get("avg_tokens"),
        "avg_latency": summary.get("avg_latency"),
        "code_failed": summary.get("extra_summary", {}).get("code_failed"),
        "run_test_failed": summary.get("extra_summary", {}).get("run_test_failed"),
    }

    overall_df = pd.DataFrame([overall])
    print("\n[1] Overall Summary")
    print(overall_df.to_string(index=False))

    # 2. Success@k curve
    success_traj = traj_df[traj_df["final_status"] == "PASS"].copy()
    success_by_call = Counter(success_traj["call_count"])

    curve_rows = []
    cumulative = 0
    for k in range(1, max_calls + 1):
        cumulative += success_by_call.get(k, 0)
        curve_rows.append({
            "method": method,
            "k": k,
            "success_count_at_k": cumulative,
            "success_rate_at_k": cumulative / total_problems,
        })

    success_curve_df = pd.DataFrame(curve_rows)
    print("\n[2] Success@k Curve")
    print(success_curve_df.to_string(index=False))

    # 3. Calls-to-success distribution
    calls_success_df = (
        success_traj["call_count"]
        .value_counts()
        .sort_index()
        .reset_index()
    )
    calls_success_df.columns = ["call_count", "num_success"]
    print("\n[3] Calls-to-Success Distribution")
    print(calls_success_df.to_string(index=False))

    # 4. Final failure breakdown
    fail_traj = traj_df[traj_df["final_status"] != "PASS"].copy()
    final_failure_df = (
        fail_traj["failure_family"]
        .value_counts()
        .reset_index()
    )
    final_failure_df.columns = ["final_failure_family", "count"]
    print("\n[4] Final Failure Breakdown")
    print(final_failure_df.to_string(index=False))

    # 5. Error stage / type breakdown from step logs
    failed_steps = step_df[step_df["status"] != "PASS"].copy()

    stage_breakdown = (
        failed_steps["error_stage"]
        .fillna("UNKNOWN")
        .value_counts()
        .reset_index()
    )
    stage_breakdown.columns = ["error_stage", "count"]

    type_breakdown = (
        failed_steps["error_type"]
        .fillna("UNKNOWN")
        .value_counts()
        .reset_index()
    )
    type_breakdown.columns = ["error_type", "count"]

    print("\n[5] Step-level Error Stage Breakdown")
    print(stage_breakdown.to_string(index=False))

    print("\n[6] Step-level Error Type Breakdown")
    print(type_breakdown.to_string(index=False))

    # 6. Transition path analysis
    path_counter = Counter(tuple(path) for path in traj_df["transition_path"])
    path_rows = []
    for path, count in path_counter.most_common(20):
        path_rows.append({
            "method": method,
            "count": count,
            "transition_path": " → ".join(path),
        })

    transition_df = pd.DataFrame(path_rows)
    print("\n[7] Top Transition Paths")
    print(transition_df.to_string(index=False))

    # 7. Normalized transition patterns
    norm_path_counter = Counter(
        tuple(normalize_status(s) for s in path)
        for path in traj_df["transition_path"]
    )

    norm_rows = []
    for path, count in norm_path_counter.most_common(20):
        norm_rows.append({
            "method": method,
            "count": count,
            "normalized_transition_path": " → ".join(path),
        })

    norm_transition_df = pd.DataFrame(norm_rows)
    print("\n[8] Top Normalized Transition Paths")
    print(norm_transition_df.to_string(index=False))

    # 8. Stagnation analysis
    stagnation_rows = []

    for _, row in traj_df.iterrows():
        path = row["transition_path"]
        norm_path = [normalize_status(s) for s in path]

        repeated_test_fail = max_consecutive(norm_path, "TEST_FAIL")
        repeated_exec_fail = max_consecutive(norm_path, "EXEC_FAIL")

        stagnation_rows.append({
            "method": method,
            "problem_id": row["problem_id"],
            "final_status": row["final_status"],
            "call_count": row["call_count"],
            "max_consecutive_test_fail": repeated_test_fail,
            "max_consecutive_exec_fail": repeated_exec_fail,
            "transition_path": " → ".join(path),
        })

    stagnation_df = pd.DataFrame(stagnation_rows)

    stagnation_summary = pd.DataFrame([{
        "method": method,
        "num_test_stagnation_ge_3": int((stagnation_df["max_consecutive_test_fail"] >= 3).sum()),
        "num_exec_stagnation_ge_3": int((stagnation_df["max_consecutive_exec_fail"] >= 3).sum()),
        "num_test_stagnation_ge_5": int((stagnation_df["max_consecutive_test_fail"] >= 5).sum()),
        "num_exec_stagnation_ge_5": int((stagnation_df["max_consecutive_exec_fail"] >= 5).sum()),
    }])

    print("\n[9] Stagnation Summary")
    print(stagnation_summary.to_string(index=False))

    # 9. Budget efficiency
    budget_rows = []

    all_avg_calls = traj_df["call_count"].mean()
    success_avg_calls = success_traj["call_count"].mean() if len(success_traj) > 0 else None
    fail_avg_calls = fail_traj["call_count"].mean() if len(fail_traj) > 0 else None

    budget_rows.append({
        "method": method,
        "avg_calls_all": all_avg_calls,
        "avg_calls_success": success_avg_calls,
        "avg_calls_failure": fail_avg_calls,
        "avg_tokens_all": traj_df["total_tokens"].mean(),
        "avg_tokens_success": success_traj["total_tokens"].mean() if len(success_traj) > 0 else None,
        "avg_tokens_failure": fail_traj["total_tokens"].mean() if len(fail_traj) > 0 else None,
    })

    budget_df = pd.DataFrame(budget_rows)
    print("\n[10] Budget Efficiency")
    print(budget_df.to_string(index=False))

    return {
        "overall": overall_df,
        "success_curve": success_curve_df,
        "calls_success": calls_success_df,
        "final_failure": final_failure_df,
        "stage_breakdown": stage_breakdown,
        "type_breakdown": type_breakdown,
        "transition_paths": transition_df,
        "normalized_transition_paths": norm_transition_df,
        "stagnation": stagnation_df,
        "stagnation_summary": stagnation_summary,
        "budget": budget_df,
    }


def max_consecutive(seq, target):
    max_len = 0
    cur = 0
    for x in seq:
        if x == target:
            cur += 1
            max_len = max(max_len, cur)
        else:
            cur = 0
    return max_len


def analyze_multiple_runs(run_dirs, output_dir="analysis_outputs"):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    all_results = defaultdict(list)

    for run_dir in run_dirs:
        result = analyze_run(run_dir)
        for key, df in result.items():
            all_results[key].append(df)

    merged = {}
    for key, dfs in all_results.items():
        merged_df = pd.concat(dfs, ignore_index=True)
        merged[key] = merged_df
        merged_df.to_csv(output_dir / f"{key}.csv", index=False)

    print("\nSaved CSV files to:", output_dir)
    return merged

In [2]:

run_dirs = [
    # "../results/qwen25coder7b/humaneval/single"
    "../results/qwen25coder7b/humaneval/repair"
    # "../results/qwen25coder7b/humaneval/code_then_plan"
    # "../results/qwen25coder7b/humaneval/code_then_plan_repair"
]

analyze_multiple_runs(run_dirs, output_dir="analysis_outputs")

Run: repair_loop

[1] Overall Summary
     method  total_problems  num_success  success_at_k  execution_success_rate  conditional_success  avg_calls  avg_tokens  avg_latency  code_failed  run_test_failed
repair_loop             164          138      0.841463                0.878049             0.958333   4.378049 3501.841463    10.589165           20                6

[2] Success@k Curve
     method  k  success_count_at_k  success_rate_at_k
repair_loop  1                 113           0.689024
repair_loop  2                 122           0.743902
repair_loop  3                 132           0.804878
repair_loop  4                 134           0.817073
repair_loop  5                 137           0.835366
repair_loop  6                 137           0.835366
repair_loop  7                 137           0.835366
repair_loop  8                 137           0.835366
repair_loop  9                 137           0.835366
repair_loop 10                 137           0.835366
repair_loop 11 

{'overall':         method  total_problems  num_success  success_at_k  \
 0  repair_loop             164          138      0.841463   
 
    execution_success_rate  conditional_success  avg_calls   avg_tokens  \
 0                0.878049             0.958333   4.378049  3501.841463   
 
    avg_latency  code_failed  run_test_failed  
 0    10.589165           20                6  ,
 'success_curve':          method   k  success_count_at_k  success_rate_at_k
 0   repair_loop   1                 113           0.689024
 1   repair_loop   2                 122           0.743902
 2   repair_loop   3                 132           0.804878
 3   repair_loop   4                 134           0.817073
 4   repair_loop   5                 137           0.835366
 5   repair_loop   6                 137           0.835366
 6   repair_loop   7                 137           0.835366
 7   repair_loop   8                 137           0.835366
 8   repair_loop   9                 137           0.8353